In [1]:
!pip install -q evaluate rouge_score sacrebleu nltk

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 10.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.8.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platfor

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

In [3]:
from huggingface_hub import login

login(token=HF_TOKEN)

In [4]:
!rm -rf /kaggle/working/event-planned-story-gen
!git clone https://github.com/abirmondal/event-planned-story-gen.git

Cloning into 'event-planned-story-gen'...
remote: Enumerating objects: 278, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (97/97), done.
remote: Total 278 (delta 72), reused 120 (delta 47), pack-reused 127 (from 1)
Receiving objects: 100% (278/278), 57.14 MiB | 21.08 MiB/s, done.
Resolving deltas: 100% (127/127), done.
Updating files: 100% (44/44), done.


In [5]:
import sys
from pathlib import Path

# Add the parent directory's path to sys.path
# sys.path requires strings, so we convert the Path object
sys.path.append(str("/kaggle/working/event-planned-story-gen"))

In [6]:
import torch
from tqdm.auto import tqdm
from src.dataset_prep.data_for_train import DataForTrain
from src.graph_construction.event_build import EventGraphBuilder
from src.utils.event_evals import events_map_to_best_graph_events
from transformers import pipeline

2025-09-03 18:30:16.876693: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1756924217.114793      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1756924217.197890      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [7]:
event_graph_read_obj = EventGraphBuilder()

event_graph = event_graph_read_obj.load_graph_pickle('event_graph_new.gpickle', load_message=False)
events_list = list(event_graph.nodes)

In [8]:
data_prep = DataForTrain(
    event_filename_suffix='_event.source_new',
    data_types=['test']
)

dataset_dict = data_prep.get_data_for_lc_to_event()

Processing test data:   0%|          | 0/4909 [00:00<?, ?it/s]

In [9]:
device_id = 0 if torch.cuda.is_available() else -1

In [10]:
HF_MODEL_NAME = "abirmondalind/lc-to-event-BART"

In [11]:
generator = pipeline(
    "text2text-generation",
    model=HF_MODEL_NAME,
    tokenizer=HF_MODEL_NAME,
    device=device_id
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

Device set to use cpu


In [12]:
# for split_name, dataset in dataset_dict.items():
#     dataset_dict[split_name] = dataset.shuffle()

In [13]:
number_of_samples = 5
# exp_dataset = dataset_dict['test'].select(range(number_of_samples))
exp_dataset = dataset_dict['test'][0,1,2,3,128,129,130,131]

In [14]:
# texts_to_test = ["Hello [EVENT_s]"]
texts_to_test = exp_dataset['source']
actual_events = exp_dataset['event']

In [15]:
predicted_events = []

for text in tqdm(texts_to_test, desc="Predicting events"):
    output = generator(
        text,
        max_new_tokens=32,
        num_beams=4,
        early_stopping=True,
    )
    predicted_events.append(output[0]['generated_text'])

Predicting events:   0%|          | 0/8 [00:00<?, ?it/s]

In [26]:
print("\n--- Pipeline Inference Results ---\n")

for i, text in enumerate(texts_to_test):
    print(f"Input {i+1}: {text}")
    if actual_events:
        print(f"Actual Event {i+1}: {actual_events[i]}")
    print(f"Generated Event {i+1}: {predicted_events[i]}")
    graph_mapped_event = events_map_to_best_graph_events([predicted_events[i]], events_list, tqdm_leave=False)
    print(f"Graph-mapped Event {i+1}: {graph_mapped_event[0]['event']}")
    print()


--- Pipeline Inference Results ---

Input 1: [MALE] was out jogging one morning . [EVENT_s]
Actual Event 1: was crisp [EVENT_e]
Generated Event 1: heard noise 


Mapping events to graph events:   0%|          | 0/1 [00:00<?, ?it/s]

Graph-mapped Event 1: heard noise

Input 2: [MALE] was out jogging one morning . [EVENT_s] was crisp [EVENT_e]
Actual Event 2: felt good [EVENT_e]
Generated Event 2: heard noise 


Mapping events to graph events:   0%|          | 0/1 [00:00<?, ?it/s]

Graph-mapped Event 2: heard noise

Input 3: [MALE] was out jogging one morning . [EVENT_s] was crisp [EVENT_sep] felt good [EVENT_e]
Actual Event 3: decided keep [EVENT_e]
Generated Event 3: ran 


...

Graph-mapped Event 6: woke up

Input 7: [FEMALE] wanted her dog to be warm in the winter . [EVENT_s] shivered [EVENT_sep] found outfit [EVENT_e]
Actual Event 7: dressed him [EVENT_e]
Generated Event 7: bought it 


Mapping events to graph events:   0%|          | 0/1 [00:00<?, ?it/s]

Graph-mapped Event 7: bought it

Input 8: [FEMALE] wanted her dog to be warm in the winter . [EVENT_s] shivered [EVENT_sep] found outfit [EVENT_sep] dressed him [EVENT_e]
Actual Event 8: n't shiver [EVENT_e]
Generated Event 8: laughed 


Mapping events to graph events:   0%|          | 0/1 [00:00<?, ?it/s]

Graph-mapped Event 8: laughed

